## 1. Dataset Choice – bank_marketing

**Goal:** predict whether a bank client will subscribe to a term deposit after a marketing call, so the bank can focus its calls on the clients most likely to say "yes".

**Dataset:** UCI Bank Marketing (#222, CC BY 4.0), full file `bank-full.csv`, saved in this project as `data/raw/bank_marketing.csv`. 45,211 clients, 16 input columns, target `y` (yes/no). Only 11.7% subscribed, so the classes are imbalanced.

**Why the full file:** the course's committed sample has only 4,500 rows and randomly generated labels (validation AUC ≈ 0.52 when tested), so I used the full UCI data.


In [40]:
import sys, pathlib
p = pathlib.Path.cwd()
while not (p / 'src').exists():
    p = p.parent
sys.path.insert(0, str(p / 'src')); REPO = p
import warnings; warnings.filterwarnings('ignore')

In [41]:
from course_utils import data
import pandas as pd

DATASET = 'bank_marketing'
TARGET = 'y'
df = data.load_raw(DATASET)
df.head()

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,no
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,no
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown,no
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown,no


## 2. Profile & Clean

### 2.1. Profile

In [42]:
# Overall inspection of the dataset
data.profile_frame(df)

shape: 45,211 rows x 17 cols | duplicated rows: 0


,dtype,missing,missing_%,unique,sample
age,int64,0,0.0,77,58
job,str,0,0.0,12,management
marital,str,0,0.0,3,married
education,str,0,0.0,4,tertiary
default,str,0,0.0,2,no
balance,int64,0,0.0,7168,2143
housing,str,0,0.0,2,yes
loan,str,0,0.0,2,no
contact,str,0,0.0,3,unknown
day,int64,0,0.0,31,5


In [43]:
# Check balance of y/n
print(df[TARGET].value_counts().to_string())
print(f"Positive rate: {df[TARGET].eq('yes').mean():.1%}")
df.describe().T.round(1)

y
no     39922
yes     5289
Positive rate: 11.7%


,count,mean,std,min,25%,50%,75%,max
age,45211.0,40.9,10.6,18.0,33.0,39.0,48.0,95.0
balance,45211.0,1362.3,3044.8,-8019.0,72.0,448.0,1428.0,102127.0
day,45211.0,15.8,8.3,1.0,8.0,16.0,21.0,31.0
duration,45211.0,258.2,257.5,0.0,103.0,180.0,319.0,4918.0
campaign,45211.0,2.8,3.1,1.0,1.0,2.0,3.0,63.0
pdays,45211.0,40.2,100.1,-1.0,-1.0,-1.0,-1.0,871.0
previous,45211.0,0.6,2.3,0.0,0.0,0.0,0.0,275.0


**Conclusion**: data is imbalanced => accuracy alone may be a misleading metric

### 2.2. Quality decisions
#### Decision 1 — Drop `duration` (target leakage)
`duration` is the duration of the last call, which is only known after the call ends. By then the outcome is known too. Therefore, it should be discarded for a predictive model.

In [44]:
buckets = pd.cut(df['duration'], [-1, 60, 120, 300, 600, 5000], labels=['< 1m','1-2m','2-5m','5-10m','> 10m'])
df.groupby(buckets, observed=True)[TARGET].apply(lambda s: f"{s.eq('yes').mean():.1%}  (n={len(s)})").to_frame('subscribe rate')

,subscribe rate
duration,
< 1m,0.2% (n=4766)
1-2m,2.2% (n=9278)
2-5m,8.6% (n=18893)
5-10m,19.2% (n=8484)
> 10m,48.4% (n=3790)


In [45]:
# How much the leak inflates performance: same model, same split, but with vs. without duration
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

def quick_score(frame):
    y = frame[TARGET].eq('yes').astype(int)
    X = pd.get_dummies(frame.drop(columns=[TARGET]), drop_first=True)
    S = data.make_splits(X, y)
    m = XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.1, random_state=42).fit(S['X_train'], S['y_train'])
    p = m.predict_proba(S['X_val'])[:, 1]
    return round(roc_auc_score(S['y_val'], p), 3), round(average_precision_score(S['y_val'], p), 3)

pd.DataFrame({
                'with duration (leak)': quick_score(df),
                'without duration (honest)': quick_score(df.drop(columns=['duration']))
             },
             index=['ROC-AUC (val)', 'PR-AUC (val)'])

,with duration (leak),without duration (honest)
ROC-AUC (val),0.935,0.806
PR-AUC (val),0.634,0.467


#### Decision 2 — Decode the `pdays = -1` sentinel
`pdays` = days since the client was last contacted in a previous campaign; -1 means "never contacted".
Left as is, a model treats -1 as a number that is just below most recent contacts, which is the opposite of reality.

In [46]:
# Check to ensure that originally -1 pdays means "never contacted"
print(f"-1 pdays: {df['pdays'].eq(-1).mean():.1%} of rows")
pd.crosstab(df['pdays'].eq(-1).rename('-1 pdays'), df['poutcome'])

-1 pdays: 81.7% of rows


poutcome,failure,other,success,unknown
-1 pdays,,,,
False,4901,1840,1511,5
True,0,0,0,36954


In the results above, we can see that when pdays is -1, there are no previous failures/successes/other responses - they have `unknown` instead, meaning the clients weren't contacted before.

**Fix:** add an explicit `prev_contacted` flag (0/1) and recode -1 to 999 ("longer ago than any real contact"; the max real value is 871). This way,
 the ordering of `pdays` is meaningful.

#### Decision 3 — Keep `"unknown"` as its own category (don't impute or drop)
There are no NaNs, but missing values are encoded as the string `"unknown"`. Dropping those rows would lose 82% of the data (because of `poutcome` which is `"unknown"` for most records since they haven't been contacted before). Imputing the most common value would erase a real signal: whether a value is unknown is itself predictive.

In [47]:
rows = []

for c in ['contact', 'education', 'job', 'poutcome']:
    unknown = df[c].eq('unknown')
    rows.append({'column': c, 'unknown %': f"{unknown.mean():.1%}",
                 'subscribe rate | unknown': f"{df.loc[unknown, TARGET].eq('yes').mean():.1%}",
                 'subscribe rate | known': f"{df.loc[~unknown, TARGET].eq('yes').mean():.1%}"})
pd.DataFrame(rows)

,column,unknown %,subscribe rate | unknown,subscribe rate | known
0,contact,28.8%,4.1%,14.8%
1,education,4.1%,13.6%,11.6%
2,job,0.6%,11.8%,11.7%
3,poutcome,81.7%,9.2%,23.1%


Clients with `contact = unknown` subscribe at ~4% vs. ~15% for known contact. That is a significant difference and doesn't seem random, so `"unknown"` stays as its own category.

#### Decision 4 — Keep the extreme values (don't clip them)
`balance`, `campaign` and `previous` have long right tails. They are plausible (e.g. wealthy clients have very high balances),
and the planned model (gradient-boosted trees) only cares about the order of values, not their scale, so clipping would give no benefit.

In [48]:
df[['balance', 'campaign', 'previous']].quantile([.5, .99, .999, 1]).round(0).astype(int)

,balance,campaign,previous
0.500,448,2,0
0.990,13165,16,9
0.999,32893,32,22
1.000,102127,63,275


In [49]:
# Check for duplicates and missing values
print("Duplicated rows:", df.duplicated().sum())
print("Missing values (NaN):", df.isna().sum().sum())

Duplicated rows: 0
Missing values (NaN): 0


#### Other checks
* **Duplicates:** 0 duplicated rows. **NaNs:** none.

### 2.3. Apply cleaning & save

In [50]:
clean = data.clean_bank(df)

# Ensuring I've preserved the shape (I dropped one column and added another one)
print(f"Original shape: {df.shape}; Cleaned shape: {clean.shape}; Columns dropped: {sorted(set(df) - set(clean))}, Added: {sorted(set(clean) - set(df))}")

# Ensuring the yes/no answers became 1/0 and the yes-rate is unchanged (11.7%)
print(clean[TARGET].value_counts(normalize=True).round(3).to_dict())
data.save_processed(clean, 'bank_clean')
clean.head()

Original shape: (45211, 17); Cleaned shape: (45211, 17); Columns dropped: ['duration'], Added: ['prev_contacted']
{0: 0.883, 1: 0.117}
saved -> /Users/alexandrinamehandzhiyska/Downloads/cos3121-ml-project/data/processed/bank_clean.parquet


,age,job,marital,education,default,balance,housing,loan,contact,day,month,campaign,pdays,previous,poutcome,y,prev_contacted
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,1,999,0,unknown,0,0
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,1,999,0,unknown,0,0
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,1,999,0,unknown,0,0
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,1,999,0,unknown,0,0
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,1,999,0,unknown,0,0


## 3. Model Training

### 3.1. Prepare Features

In [51]:
# Separate y column (the output variable) from the input columns
y = clean['y']
X = clean.drop(columns=['y'])

In [52]:
# Turn text data into numerical data
X = pd.get_dummies(X, drop_first=True, dtype=int)

### 3.2. Train / validation / test split

In [53]:
# Split data into training, validation, and test groups
S = data.make_splits(X, y)

In [54]:
for name in ['train', 'val', 'test']:
    print(name.capitalize(), S['X_' + name].shape, f"Yes rate: {S['y_' + name].mean():.1%}")


Train (31647, 42) Yes rate: 11.7%
Val (6782, 42) Yes rate: 11.7%
Test (6782, 42) Yes rate: 11.7%


### 3.3. Baseline: always predict "no"

In [55]:
# Use a dumb model as a base benchmark - always predict "no" (the most frequent answer)

from sklearn.dummy import DummyClassifier
from course_utils import viz

dummy = DummyClassifier(strategy='most_frequent')
dummy.fit(S['X_train'], S['y_train'])
dummy_probs = dummy.predict_proba(S['X_val'])[:, 1]
viz.metric_report(S['y_val'], dummy_probs, name='dummy: always "no" (val)')

--- dummy: always "no" (val) ---
  accuracy: 0.883
 precision: 0.000
    recall: 0.000
        f1: 0.000
   auc_roc: 0.500


{'accuracy': 0.8829253907401946,
 'precision': 0.0,
 'recall': 0.0,
 'f1': 0.0,
 'auc_roc': 0.5}

### 3.4. Simple model: logistic regression

In [56]:
# Try with logistic regression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

logreg = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
logreg.fit(S['X_train'], S['y_train'])
logreg_probs = logreg.predict_proba(S['X_val'])[:, 1]
viz.metric_report(S['y_val'], logreg_probs, name='logistic regression (val)')

--- logistic regression (val) ---
  accuracy: 0.893
 precision: 0.658
    recall: 0.181
        f1: 0.284
   auc_roc: 0.770


{'accuracy': 0.8930993807136538,
 'precision': 0.6575342465753424,
 'recall': 0.181360201511335,
 'f1': 0.2843040473840079,
 'auc_roc': 0.7698638250472397}

### 3.5. Main model: XGBoost

In [57]:
from xgboost import XGBClassifier

xgb = XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.1, eval_metric='auc', random_state=42)
xgb.fit(S['X_train'], S['y_train'])
xgb_probs = xgb.predict_proba(S['X_val'])[:, 1]
viz.metric_report(S['y_val'], xgb_probs, name='XGBoost (val)')


--- XGBoost (val) ---
  accuracy: 0.895
 precision: 0.650
    recall: 0.232
        f1: 0.342
   auc_roc: 0.808


{'accuracy': 0.8954585667944559,
 'precision': 0.6501766784452296,
 'recall': 0.23173803526448364,
 'f1': 0.34168987929433614,
 'auc_roc': 0.808413005692325}

### 3.6. Precision/recall at different cut-offs

In [58]:
from sklearn.metrics import precision_score, recall_score, average_precision_score

rows = []

for cutoff in [0.5, 0.4, 0.3, 0.2, 0.15, 0.1]:
    decision = (xgb_probs >= cutoff).astype(int)

    rows.append({
        'cut-off': cutoff,
        'people called': decision.sum(),
        'subscribers found': (decision * S['y_val']).sum(),
        'precision': round(precision_score(S['y_val'], decision), 3),
        'recall': round(recall_score(S['y_val'], decision), 3),
    })

print(f"PR-AUC (val): {average_precision_score(S['y_val'], xgb_probs):.3f}   (random guessing = {S['y_val'].mean():.3f})")
pd.DataFrame(rows)

PR-AUC (val): 0.470   (random guessing = 0.117)


,cut-off,people called,subscribers found,precision,recall
0,0.50,283,184,0.650,0.232
1,0.40,412,250,0.607,0.315
2,0.30,620,322,0.519,0.406
3,0.20,917,432,0.471,0.544
4,0.15,1202,486,0.404,0.612
5,0.10,2039,563,0.276,0.709


## 4. Hyperparameter tuning
### 4.1. max_depth

In [59]:
from sklearn.metrics import roc_auc_score

rows = []
for depth in [2, 3, 4, 6, 8]:
    model = XGBClassifier(n_estimators=200, max_depth=depth, learning_rate=0.1, eval_metric='auc', random_state=42)
    model.fit(S['X_train'], S['y_train'])
    train_probs = model.predict_proba(S['X_train'])[:, 1]
    val_probs = model.predict_proba(S['X_val'])[:, 1]

    rows.append({
        'max_depth': depth,
        'train AUC': round(roc_auc_score(S['y_train'], train_probs), 3),
        'val AUC': round(roc_auc_score(S['y_val'], val_probs), 3),
        'val PR-AUC': round(average_precision_score(S['y_val'], val_probs), 3),
    })

depth_table = pd.DataFrame(rows)
depth_table['gap'] = (depth_table['train AUC'] - depth_table['val AUC']).round(3)
depth_table

,max_depth,train AUC,val AUC,val PR-AUC,gap
0,2,0.804,0.802,0.451,0.002
1,3,0.822,0.807,0.466,0.015
2,4,0.844,0.808,0.470,0.036
3,6,0.896,0.806,0.455,0.090
4,8,0.951,0.798,0.459,0.153


### 4.2. learning_rate

In [60]:
rows = []

for lr in [0.01, 0.03, 0.1, 0.3, 0.5]:                               
    model = XGBClassifier(n_estimators=200, max_depth=4, learning_rate=lr, eval_metric='auc', random_state=42)
    model.fit(S['X_train'], S['y_train'])
    train_probs = model.predict_proba(S['X_train'])[:, 1]
    val_probs = model.predict_proba(S['X_val'])[:, 1]

    rows.append({
        'learning_rate': lr,                                        
        'train AUC': round(roc_auc_score(S['y_train'], train_probs), 3),
        'val AUC': round(roc_auc_score(S['y_val'], val_probs), 3),
        'val PR-AUC': round(average_precision_score(S['y_val'], val_probs), 3),
    })

lr_table = pd.DataFrame(rows)
lr_table['gap'] = (lr_table['train AUC'] - lr_table['val AUC']).round(3)
lr_table

,learning_rate,train AUC,val AUC,val PR-AUC,gap
0,0.01,0.784,0.777,0.432,0.007
1,0.03,0.813,0.800,0.457,0.013
2,0.10,0.844,0.808,0.470,0.036
3,0.30,0.886,0.799,0.443,0.087
4,0.50,0.911,0.787,0.415,0.124


### 4.3. Grid search: max_depth x learning_rate

In [61]:
rows = []

for depth in [3, 4, 5, 6]:
    for lr in [0.03, 0.05, 0.1, 0.2]:
        model = XGBClassifier(n_estimators=200, max_depth=depth, learning_rate=lr, eval_metric='auc', random_state=42)
        model.fit(S['X_train'], S['y_train'])
        val_probs = model.predict_proba(S['X_val'])[:, 1]

        rows.append({
            'max_depth': depth,
            'learning_rate': lr,
            'val AUC': round(roc_auc_score(S['y_val'], val_probs), 4),
            'val PR-AUC': round(average_precision_score(S['y_val'], val_probs), 4),
        })

grid = pd.DataFrame(rows)
display(grid.pivot(index='max_depth', columns='learning_rate', values='val AUC'))
grid.sort_values('val AUC', ascending=False)

learning_rate,0.03,0.05,0.10,0.20
max_depth,,,,
3,0.7923,0.8019,0.8071,0.8042
4,0.8004,0.8059,0.8084,0.8040
5,0.8035,0.8087,0.8091,0.8025
6,0.8075,0.8113,0.8062,0.7881


,max_depth,learning_rate,val AUC,val PR-AUC
13,6,0.05,0.8113,0.4648
10,5,0.10,0.8091,0.4646
9,5,0.05,0.8087,0.4692
6,4,0.10,0.8084,0.4705
12,6,0.03,0.8075,0.4637
2,3,0.10,0.8071,0.4663
14,6,0.10,0.8062,0.4551
5,4,0.05,0.8059,0.4678
3,3,0.20,0.8042,0.4591
7,4,0.20,0.8040,0.4551


### 4.4. Choosing the cut-off (best F1)

In [62]:
from sklearn.metrics import f1_score

rows = []

for cutoff in [0.1, 0.15, 0.2, 0.25, 0.3, 0.4, 0.5]:
    decision = (xgb_probs >= cutoff).astype(int)
    rows.append({'cut-off': cutoff,
                 'precision': round(precision_score(S['y_val'], decision), 3),
                 'recall': round(recall_score(S['y_val'], decision), 3),
                 'F1': round(f1_score(S['y_val'], decision), 3)})
f1_table = pd.DataFrame(rows)
display(f1_table)

CUTOFF = f1_table.loc[f1_table['F1'].idxmax(), 'cut-off']
print('Chosen cut-off:', CUTOFF)

,cut-off,precision,recall,F1
0,0.10,0.276,0.709,0.397
1,0.15,0.404,0.612,0.487
2,0.20,0.471,0.544,0.505
3,0.25,0.500,0.475,0.487
4,0.30,0.519,0.406,0.455
5,0.40,0.607,0.315,0.415
6,0.50,0.650,0.232,0.342


Chosen cut-off: 0.2


**Decision:** I keep a cut-off of **0.2**. It has the highest F1 on the validation set (0.505: precision 0.471, recall 0.544), so it gives the best balance between wasted calls and missed subscribers. With real call costs and customer values, I would choose the cut-off that maximises expected profit instead of F1.

### 4.5. Final evaluation on the test set (used once)

In [63]:
test_probs = xgb.predict_proba(S['X_test'])[:, 1]
final = viz.metric_report(S['y_test'], test_probs, threshold=CUTOFF, name=f'FINAL XGBoost, cut-off {CUTOFF} (test)')
print(f"    pr_auc: {average_precision_score(S['y_test'], test_probs):.3f}")

--- FINAL XGBoost, cut-off 0.2 (test) ---
  accuracy: 0.877
 precision: 0.477
    recall: 0.523
        f1: 0.499
   auc_roc: 0.798
    pr_auc: 0.468


## 5. Deployment
### 5.1. Save the model

In [27]:
import json

MODEL_DIR = REPO / 'models'
xgb.save_model(str(MODEL_DIR / 'phase1_bank_marketing.json'))

meta = {
    'features': X.columns.tolist(),
    'cutoff': float(CUTOFF),
    'test_metrics': {name: round(float(value), 4) for name, value in final.items()},
}

(MODEL_DIR / 'phase1_bank_marketing_meta.json').write_text(json.dumps(meta, indent=2))
print('Saved model + metadata to', MODEL_DIR)


Saved model + metadata to /Users/alexandrinamehandzhiyska/Downloads/cos3121-ml-project/models


### 5.2. Test the serving script locally

In [28]:
import sys
sys.path.insert(0, str(REPO / 'project' / 'serving'))
import inference

bundle = inference.model_fn(str(MODEL_DIR))
sample = S['X_test'].iloc[[0, int(test_probs.argmax())]]
body = json.dumps({'instances': sample.to_dict(orient='records')})
response = inference.output_fn(inference.predict_fn(inference.input_fn(body, 'application/json'), bundle))

print('Simulated endpoint:', response)
print('Notebook model    :', xgb.predict_proba(sample)[:, 1].round(4))

Simulated endpoint: {"predictions": [{"subscribe_probability": 0.0393, "call": false}, {"subscribe_probability": 0.9388, "call": true}]}
Notebook model    : [0.0393 0.9388]


### 5.3. Package and upload to S3

In [29]:
import tarfile, boto3

REGION = 'us-east-1'
ACCOUNT = boto3.client('sts').get_caller_identity()['Account']
BUCKET = f'sagemaker-{REGION}-{ACCOUNT}'
s3 = boto3.client('s3', region_name=REGION)

if BUCKET not in [b['Name'] for b in s3.list_buckets()['Buckets']]:
    s3.create_bucket(Bucket=BUCKET)

package = MODEL_DIR / 'phase1_bank_marketing.tar.gz'

with tarfile.open(package, 'w:gz') as tar:
    tar.add(MODEL_DIR / 'phase1_bank_marketing.json', arcname='phase1_bank_marketing.json')
    tar.add(MODEL_DIR / 'phase1_bank_marketing_meta.json', arcname='phase1_bank_marketing_meta.json')
    tar.add(REPO / 'project' / 'serving' / 'inference.py', arcname='code/inference.py')
    tar.add(REPO / 'project' / 'serving' / 'requirements.txt', arcname='code/requirements.txt')

MODEL_S3_KEY = 'phase1/phase1_bank_marketing.tar.gz'
s3.upload_file(str(package), BUCKET, MODEL_S3_KEY)
MODEL_S3_URI = f's3://{BUCKET}/{MODEL_S3_KEY}'
print('Uploaded ->', MODEL_S3_URI)

Uploaded -> s3://sagemaker-us-east-1-008300324202/phase1/phase1_bank_marketing.tar.gz


### 5.4. Create the SageMaker endpoint

In [35]:
from sagemaker.core import image_uris

ENDPOINT_NAME = 'phase1-bank-marketing'
ROLE_ARN = f'arn:aws:iam::{ACCOUNT}:role/SageMakerCourseRole'
IMAGE_URI = image_uris.retrieve('sklearn', region=REGION, version='1.9-0',
                                image_scope='inference', instance_type='ml.m5.large')
print('Server software:', IMAGE_URI)

sm = boto3.client('sagemaker', region_name=REGION)

sm.create_model(
    ModelName=ENDPOINT_NAME,
    ExecutionRoleArn=ROLE_ARN,
    PrimaryContainer={
        'Image': IMAGE_URI,
        'ModelDataUrl': MODEL_S3_URI,
        'Environment': {
            'SAGEMAKER_PROGRAM': 'inference.py',
            'SAGEMAKER_SUBMIT_DIRECTORY': '/opt/ml/model/code',
            'SAGEMAKER_REGION': REGION,
        },
    },
)
sm.create_endpoint_config(
    EndpointConfigName=ENDPOINT_NAME,
    ProductionVariants=[{'VariantName': 'AllTraffic', 'ModelName': ENDPOINT_NAME,
                         'InstanceType': 'ml.m5.large', 'InitialInstanceCount': 1}],
)
sm.create_endpoint(EndpointName=ENDPOINT_NAME, EndpointConfigName=ENDPOINT_NAME)
print('Endpoint is being created - billing has started. Waiting until it is ready (5-10 min)...')

sm.get_waiter('endpoint_in_service').wait(EndpointName=ENDPOINT_NAME)
print('Status:', sm.describe_endpoint(EndpointName=ENDPOINT_NAME)['EndpointStatus'])

Server software: 683313688378.dkr.ecr.us-east-1.amazonaws.com/sagemaker-scikit-learn:1.9-0-cpu-py3
Endpoint is being created - billing has started. Waiting until it is ready (5-10 min)...
Status: InService


### 5.5. Invoke the endpoint

In [36]:
runtime = boto3.client('sagemaker-runtime', region_name=REGION)
reply = runtime.invoke_endpoint(EndpointName=ENDPOINT_NAME, ContentType='application/json', Body=body)

print('Endpoint answer:', reply['Body'].read().decode())
print('Notebook model:', xgb.predict_proba(sample)[:, 1].round(4))

Endpoint answer: {"predictions": [{"subscribe_probability": 0.0393, "call": false}, {"subscribe_probability": 0.9388, "call": true}]}
Notebook model: [0.0393 0.9388]


### 5.6. Delete the endpoint (stop billing)

In [33]:
sm.delete_endpoint(EndpointName=ENDPOINT_NAME)
sm.delete_endpoint_config(EndpointConfigName=ENDPOINT_NAME)
sm.delete_model(ModelName=ENDPOINT_NAME)
print('Deleted endpoint, endpoint config and model:', ENDPOINT_NAME)

Deleted endpoint, endpoint config and model: phase1-bank-marketing
